# MODULE 3 : Développement Applicatif
## VI : Gestion des bases de données (MySQL)

**Rappel du contexte :** Toute la semaine, vos données (clients, commandes) vivaient **en mémoire**, elles disparaissaient à chaque fermeture du programme. Ce n'est évidemment pas acceptable pour une vraie application d'entreprise. Aujourd'hui, on connecte l'application à une **vraie base de données MySQL**, le système utilisé par une immense majorité d'entreprises pour ce type d'applications de gestion.

## Prérequis technique

Ce notebook suppose qu'un **serveur MySQL** (ou MariaDB, compatible) est installé et démarré sur votre machin, ainsi que la bibliothèque `mysql-connector-python` :
```bash
pip install mysql-connector-python
```


### Objectifs du jour
- Comprendre le passage du MCD (Jour 1) à des tables SQL réelles
- Se connecter à MySQL depuis Python
- Réaliser les opérations CRUD (Create, Read, Update, Delete) en SQL
- Remplacer le stockage "en mémoire" de l'application par la base de données

### Déroulé de la journée
1. Du MCD aux tables SQL : création de la base
2. Connexion Python ↔ MySQL 
3. Les opérations CRUD en SQL
4. Atelier : brancher `GestionCommerciale` sur MySQL
5. Exercice noté

---
## 1. Du MCD aux tables SQL

Rappelez-vous le MCD du **Jour 1** :
```
CLIENT (id_client, nom, email, telephone)
COMMANDE (id_commande, date, produit, quantite, prix_unitaire, #id_client)
```

On traduit ça directement en SQL. Chaque `#id_client` devient une **clé étrangère** (`FOREIGN KEY`), le lien entre les deux tables.

### Script de création de la base (à exécuter une seule fois)

Ouvrez un client MySQL (ligne de commande `mysql`, ou un outil comme DBeaver/phpMyAdmin fourni par votre centre) et exécutez :

```sql
CREATE DATABASE IF NOT EXISTS gestion_commerciale;
USE gestion_commerciale;

CREATE TABLE clients (
    id_client INT AUTO_INCREMENT PRIMARY KEY,
    nom VARCHAR(100) NOT NULL,
    email VARCHAR(150) NOT NULL UNIQUE,
    telephone VARCHAR(10) NOT NULL
);

CREATE TABLE commandes (
    id_commande INT AUTO_INCREMENT PRIMARY KEY,
    id_client INT NOT NULL,
    date_commande DATE NOT NULL,
    produit VARCHAR(150) NOT NULL,
    quantite INT NOT NULL,
    prix_unitaire DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (id_client) REFERENCES clients(id_client)
);


---
## 2. Connexion Python ↔ MySQL

On utilise la bibliothèque `mysql-connector-python`, le connecteur officiel MySQL pour Python.

In [ ]:
import mysql.connector
from mysql.connector import Error

# Paramètres de connexion — à adapter selon votre environnement de formation
CONFIG_BD = {
    "host": "localhost",
    "user": "root",
    "password": "",   
    "database": "gestion_commerciale",
}

def obtenir_connexion():
    """
    Ouvre et retourne une connexion à la base de données.
    Lève une exception explicite si la connexion échoue.
    """
    try:
        connexion = mysql.connector.connect(**CONFIG_BD)
        return connexion
    except Error as erreur:
        raise ConnectionError(f"Impossible de se connecter à MySQL : {erreur}")



try:
    connexion = obtenir_connexion()
    print("Connexion réussie à la base :", CONFIG_BD["database"])
    connexion.close()
except ConnectionError as erreur:
    print(f"{erreur}")
    print("Vérifiez que MySQL est démarré et que les identifiants du CONFIG_BD sont corrects.")

---
## 3. Les opérations CRUD en SQL

**CRUD** = Create, Read, Update, Delete, les 4 opérations de base sur des données, que l'on retrouve dans quasiment toute application de gestion.

### a) CREATE : insérer un client

In [ ]:
def ajouter_client_bd(nom, email, telephone):
    """
    Insère un nouveau client dans la base de données.
    Retourne l'id du client créé.
    """
    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor()
        # IMPORTANT : on utilise des paramètres (%s), JAMAIS de f-string dans une requête SQL !
        # Sinon on s'expose à une faille d'injection SQL, une des vulnérabilités les plus
        # connues et les plus graves en développement applicatif.
        requete = "INSERT INTO clients (nom, email, telephone) VALUES (%s, %s, %s)"
        curseur.execute(requete, (nom, email, telephone))
        connexion.commit()  # valide définitivement l'insertion
        return curseur.lastrowid
    finally:
        connexion.close()  # toujours fermer la connexion, même en cas d'erreur


print("Fonction définie. Exemple d'appel (nécessite une connexion active) :")
print('ajouter_client_bd("Amina Traoré", "amina@example.com", "0612345678")')

### b) READ : lire des clients

In [ ]:
def lister_clients_bd():
    """Retourne la liste de tous les clients sous forme de liste de dictionnaires."""
    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor(dictionary=True)  # dictionary=True -> résultats sous forme de dict
        curseur.execute("SELECT * FROM clients")
        return curseur.fetchall()
    finally:
        connexion.close()


def rechercher_client_bd(id_client):
    """Retourne un client par son id, ou None s'il n'existe pas."""
    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor(dictionary=True)
        curseur.execute("SELECT * FROM clients WHERE id_client = %s", (id_client,))
        return curseur.fetchone()
    finally:
        connexion.close()


def total_depense_client_bd(id_client):
    """
    Calcule le total dépensé par un client directement en SQL (plus efficace
    que de tout récupérer en Python et calculer ensuite, surtout sur un gros volume).
    """
    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor()
        requete = """
            SELECT COALESCE(SUM(quantite * prix_unitaire), 0)
            FROM commandes
            WHERE id_client = %s
        """
        curseur.execute(requete, (id_client,))
        return curseur.fetchone()[0]
    finally:
        connexion.close()


print("Fonctions de lecture définies (lister_clients_bd, rechercher_client_bd, total_depense_client_bd).")

### c) UPDATE : modifier un client

In [ ]:
def modifier_client_bd(id_client, nom=None, email=None, telephone=None):
    """Met à jour les champs fournis d'un client existant."""
    champs_a_modifier = []
    valeurs = []

    if nom is not None:
        champs_a_modifier.append("nom = %s")
        valeurs.append(nom)
    if email is not None:
        champs_a_modifier.append("email = %s")
        valeurs.append(email)
    if telephone is not None:
        champs_a_modifier.append("telephone = %s")
        valeurs.append(telephone)

    if not champs_a_modifier:
        return  # rien à modifier

    valeurs.append(id_client)
    requete = f"UPDATE clients SET {', '.join(champs_a_modifier)} WHERE id_client = %s"

    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor()
        curseur.execute(requete, tuple(valeurs))
        connexion.commit()
        if curseur.rowcount == 0:
            raise ValueError(f"Client inconnu (id={id_client}).")
    finally:
        connexion.close()


print("Fonction modifier_client_bd définie.")

### d) DELETE : supprimer un client (avec la même règle métier que le Jour 5)

In [ ]:
def supprimer_client_bd(id_client):
    """
    Supprime un client, sauf s'il a des commandes.
    Ici, la contrainte FOREIGN KEY de la base empêcherait de toute façon
    la suppression si des commandes existent -> on vérifie d'abord côté Python
    pour donner un message d'erreur clair, plutôt que de laisser MySQL renvoyer
    une erreur technique brute à l'utilisateur.
    """
    connexion = obtenir_connexion()
    try:
        curseur = connexion.cursor()
        curseur.execute("SELECT COUNT(*) FROM commandes WHERE id_client = %s", (id_client,))
        nb_commandes = curseur.fetchone()[0]
        if nb_commandes > 0:
            raise ValueError(
                f"Impossible de supprimer ce client : {nb_commandes} commande(s) associée(s)."
            )

        curseur.execute("DELETE FROM clients WHERE id_client = %s", (id_client,))
        connexion.commit()
        if curseur.rowcount == 0:
            raise ValueError(f"Client inconnu (id={id_client}).")
    finally:
        connexion.close()


print("Fonction supprimer_client_bd définie.")

---
## 4. `GestionCommerciale` sur MySQL

On réécrit maintenant `gestion_commerciale.py` pour que **toutes les opérations passent par la base de données**, au lieu du dictionnaire en mémoire. Remarquez que **l'interface Tkinter (`interface_gestion.py`) n'a quasiment rien à changer**, c'est tout l'intérêt d'avoir séparé logique métier et interface depuis le Jour 3 !

In [ ]:


import re
import mysql.connector
from mysql.connector import Error

CONFIG_BD = {
    "host": "localhost",
    "user": "root",
    "password": "",
    "database": "gestion_commerciale",
}

def valider_email(email):
    motif = r"^[\.\-]+@[\w\-]+\.[a-zA-Z]{2,}$"
    return re.match(motif, email) is not None


def valider_telephone(telephone):
    return telephone.isdigit() and len(telephone) == 10


class ClientBD:
    """Représentation simple d'un client lu depuis la base (pas de logique, juste les données)."""
    def __init__(self, id_client, nom, email, telephone, total_depense=0):
        self.id_client = id_client
        self.nom = nom
        self.email = email
        self.telephone = telephone
        self.total_depense_valeur = total_depense

    def total_depense(self):
        return float(self.total_depense_valeur)

    def __repr__(self):
        return f"Client(id={self.id_client}, nom='{self.nom}')"


class GestionCommercialeBD:
    """Même interface publique que GestionCommerciale (Jour 3-5), mais persistée en MySQL."""

    def __init__(self, config_bd=None):
        self.config_bd = config_bd or CONFIG_BD

    def _connexion(self):
        try:
            return mysql.connector.connect(**self.config_bd)
        except Error as erreur:
            raise ConnectionError(f"Impossible de se connecter à MySQL : {erreur}")

    def ajouter_client(self, nom, email, telephone):
        if not nom or not nom.strip():
            raise ValueError("Le nom du client est obligatoire.")
        if not valider_email(email):
            raise ValueError(f"Email invalide : {email}")
        if not valider_telephone(telephone):
            raise ValueError(f"Téléphone invalide : {telephone}")

        connexion = self._connexion()
        try:
            curseur = connexion.cursor()
            try:
                curseur.execute(
                    "INSERT INTO clients (nom, email, telephone) VALUES (%s, %s, %s)",
                    (nom.strip(), email, telephone)
                )
                connexion.commit()
            except mysql.connector.IntegrityError:
                # La contrainte UNIQUE sur email a bloqué l'insertion -> doublon
                raise ValueError(f"Un client avec l'email {email} existe déjà.")
            return curseur.lastrowid
        finally:
            connexion.close()

    def ajouter_commande(self, id_client, date, produit, quantite, prix_unitaire):
        if quantite <= 0:
            raise ValueError("La quantité doit être strictement positive.")
        if prix_unitaire < 0:
            raise ValueError("Le prix ne peut pas être négatif.")

        connexion = self._connexion()
        try:
            curseur = connexion.cursor()
            try:
                curseur.execute(
                    """INSERT INTO commandes (id_client, date_commande, produit, quantite, prix_unitaire)
                       VALUES (%s, %s, %s, %s, %s)""",
                    (id_client, date, produit, quantite, prix_unitaire)
                )
                connexion.commit()
            except mysql.connector.IntegrityError:
                # La contrainte FOREIGN KEY a bloqué l'insertion -> client inconnu
                raise ValueError(f"Client inconnu (id={id_client}).")
            return curseur.lastrowid
        finally:
            connexion.close()

    def lister_clients(self):
        connexion = self._connexion()
        try:
            curseur = connexion.cursor(dictionary=True)
            curseur.execute("""
                SELECT c.id_client, c.nom, c.email, c.telephone,
                       COALESCE(SUM(cmd.quantite * cmd.prix_unitaire), 0) AS total_depense
                FROM clients c
                LEFT JOIN commandes cmd ON cmd.id_client = c.id_client
                GROUP BY c.id_client, c.nom, c.email, c.telephone
            """)
            return [ClientBD(**ligne) for ligne in curseur.fetchall()]
        finally:
            connexion.close()

    def rechercher_clients(self, terme):
        terme = f"%{terme.strip()}%"
        connexion = self._connexion()
        try:
            curseur = connexion.cursor(dictionary=True)
            curseur.execute("""
                SELECT c.id_client, c.nom, c.email, c.telephone,
                       COALESCE(SUM(cmd.quantite * cmd.prix_unitaire), 0) AS total_depense
                FROM clients c
                LEFT JOIN commandes cmd ON cmd.id_client = c.id_client
                WHERE c.nom LIKE %s OR c.email LIKE %s
                GROUP BY c.id_client, c.nom, c.email, c.telephone
            """, (terme, terme))
            return [ClientBD(**ligne) for ligne in curseur.fetchall()]
        finally:
            connexion.close()

    def supprimer_client(self, id_client):
        connexion = self._connexion()
        try:
            curseur = connexion.cursor()
            curseur.execute("SELECT COUNT(*) FROM commandes WHERE id_client = %s", (id_client,))
            if curseur.fetchone()[0] > 0:
                raise ValueError("Impossible de supprimer ce client : des commandes lui sont associées.")
            curseur.execute("DELETE FROM clients WHERE id_client = %s", (id_client,))
            connexion.commit()
            if curseur.rowcount == 0:
                raise ValueError(f"Client inconnu (id={id_client}).")
        finally:
            connexion.close()

    def exporter_clients_csv(self, chemin_fichier):
        import csv
        with open(chemin_fichier, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["ID", "Nom", "Email", "Téléphone", "Total dépensé (€)"])
            for client in self.lister_clients():
                writer.writerow([client.id_client, client.nom, client.email,
                                  client.telephone, client.total_depense()])

In [ ]:
pip install PyInstaller

### Test complet (si  serveur MySQL est disponible)

Exécutez la cellule suivante pour vérifier que tout fonctionne de bout en bout. Adaptez d'abord `CONFIG_BD` avec vos identifiants.

In [3]:
import sys
sys.path.insert(0, ".")
from gestion_commerciale_bd import GestionCommercialeBD

try:
    gestion_bd = GestionCommercialeBD()

    id_client = gestion_bd.ajouter_client("Test Notebook", "test.notebook@example.com", "0611223344")
    print("Client créé, id =", id_client)

    gestion_bd.ajouter_commande(id_client, "2026-08-23", "Souris", 1, 20.0)
    print("Commande créée.")

    print("\nListe des clients :")
    for client in gestion_bd.lister_clients():
        print(" -", client, "| total dépensé :", client.total_depense(), "€")

except ConnectionError as erreur:
    print(f"{erreur}")
    print("Ce test nécessite un serveur MySQL actif et un CONFIG_BD correct.")
    print("Demandez les identifiants à votre formateur si vous êtes en salle.")

ModuleNotFoundError: No module named 'mysql'